# Phase 6 — Evidence, Interpretation and Reporting

This notebook turns the locked Phase 4 metrics and Phase 5 rankings into evidence-backed explanations for the 20 official selections. It does not alter eligibility, scores, format combination, or ranking order.

In [1]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd

ROOT = Path.cwd()
PHASE3_DIR = ROOT / 'phase3_outputs'
PHASE4_DIR = ROOT / 'phase4_outputs'
PHASE5_DIR = ROOT / 'phase5_outputs'
OUTPUT_DIR = ROOT / 'phase6_outputs'
REPORT_PATH = ROOT / 'Phase 6 Evidence Interpretation and Reporting Report.md'
MODEL_VERSION = 'evidence_reporting_v1'
FORMATS = ['IPL', 'T20I']
BOWLER_WICKET_TYPES = {'Caught', 'Bowled', 'LBW', 'Stump', 'Hit Wicket'}
BAT_RELIABLE = {'high_records': 80, 'high_matches': 8, 'ordinary_records': 160}
BOWL_RELIABLE = {'high_records': 100, 'high_matches': 8, 'ordinary_records': 200, 'high_clean_records': 80, 'ordinary_clean_records': 160}
BAT_COMPONENTS = {
    'scoring_quality_z': ('scoring quality', 0.45),
    'survival_quality_z': ('survival quality', 0.25),
    'scoring_pressure_gradient_z': ('scoring pressure gradient', 0.20),
    'survival_pressure_gradient_z': ('survival pressure gradient', 0.10),
}
BOWL_COMPONENTS = {
    'wicket_quality_z': ('wicket quality', 0.45),
    'run_suppression_quality_z': ('run suppression quality', 0.30),
    'wicket_pressure_gradient_z': ('wicket pressure gradient', 0.15),
    'run_suppression_pressure_gradient_z': ('run suppression pressure gradient', 0.10),
}

def require(condition, message):
    if not bool(condition):
        raise ValueError(message)

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def verify_manifest(directory, manifest_name):
    manifest = pd.read_csv(directory / manifest_name)
    failures = []
    for row in manifest.itertuples(index=False):
        path = directory / row.artifact
        if not path.exists() or sha256(path) != row.sha256:
            failures.append(row.artifact)
    require(not failures, f'Manifest mismatch: {failures}')
    return manifest

def markdown_table(frame, float_digits=4):
    columns = [str(c) for c in frame.columns]
    lines = ['| ' + ' | '.join(columns) + ' |', '| ' + ' | '.join(['---'] * len(columns)) + ' |']
    for row in frame.itertuples(index=False, name=None):
        values = []
        for value in row:
            if pd.isna(value):
                values.append('—')
            elif isinstance(value, (float, np.floating)):
                values.append(f'{value:.{float_digits}f}')
            else:
                values.append(str(value))
        lines.append('| ' + ' | '.join(values) + ' |')
    return '\n'.join(lines)

phase3_manifest = verify_manifest(PHASE3_DIR, 'phase3_output_manifest.csv')
phase4_manifest = verify_manifest(PHASE4_DIR, 'phase4_output_manifest.csv')
phase5_manifest = verify_manifest(PHASE5_DIR, 'phase5_output_manifest.csv')
phase3_gates = pd.read_csv(PHASE3_DIR / 'phase3_validation_gates.csv')
phase4_gates = pd.read_csv(PHASE4_DIR / 'phase4_validation_gates.csv')
phase5_gates = pd.read_csv(PHASE5_DIR / 'phase5_validation_gates.csv')
require(phase3_gates['status'].eq('PASS').all(), 'Phase 3 validation failure')
require(phase4_gates['status'].eq('PASS').all(), 'Phase 4 validation failure')
require(phase5_gates['status'].eq('PASS').all(), 'Phase 5 validation failure')

deliveries = pd.read_csv(PHASE3_DIR / 'phase3_structural_pressure_labelled.csv', low_memory=False)
bat_format = pd.read_csv(PHASE4_DIR / 'phase4_batter_format_metrics.csv')
bowl_format = pd.read_csv(PHASE4_DIR / 'phase4_bowler_format_metrics.csv')
outcome_benchmarks = pd.read_csv(PHASE4_DIR / 'phase4_outcome_benchmarks.csv')
official_all = pd.read_csv(PHASE5_DIR / 'phase5_official_rankings.csv')
bootstrap_summary = pd.read_csv(PHASE5_DIR / 'phase5_bootstrap_summary.csv')
scenario_rankings = pd.read_csv(PHASE5_DIR / 'phase5_scenario_rankings.csv')
self_influence = pd.read_csv(PHASE5_DIR / 'phase5_self_influence.csv')
boundary_source = pd.read_csv(PHASE5_DIR / 'phase5_boundary_and_provisional_candidates.csv')
official = official_all.loc[official_all['official_top10']].copy().sort_values(['role', 'rank']).reset_index(drop=True)
require(official.groupby('role').size().eq(10).all(), 'Official selection is not 10 per role')
print(f'Loaded {len(deliveries):,} deliveries and {len(official)} official selections.')

Loaded 188,330 deliveries and 20 official selections.


## Format eligibility and score contribution evidence

In [2]:
format_metrics = pd.concat([bat_format, bowl_format], ignore_index=True, sort=False)
selected_formats = (
    official[['role', 'player_id', 'player_name', 'rank', 'qualifying_formats', 'qualifying_format_count', 'ipl_rating', 't20i_rating', 'combined_rating']]
    .assign(_join=1)
    .merge(pd.DataFrame({'format_group': FORMATS, '_join': 1}), on='_join')
    .drop(columns='_join')
)
format_evidence = selected_formats.merge(
    format_metrics.drop(columns=['player_name'], errors='ignore'),
    on=['role', 'player_id', 'format_group'], how='left', validate='one_to_one'
)
format_evidence['evidence_status'] = format_evidence['eligibility'].fillna('No observed records')
format_evidence['used_in_official_score'] = format_evidence.apply(
    lambda row: row['format_group'] in str(row['qualifying_formats']).split('+'), axis=1
)

def failed_reliable_gates(row):
    if row['evidence_status'] == 'No observed records':
        return 'no player-format performance record in supplied analytical population'
    thresholds = BAT_RELIABLE if row['role'] == 'Batter' else BOWL_RELIABLE
    failures = []
    for key, threshold in thresholds.items():
        value = row.get(key, np.nan)
        if pd.isna(value) or value < threshold:
            rendered = 'missing' if pd.isna(value) else str(int(value))
            failures.append(f'{key} {rendered} < {threshold}')
    return '; '.join(failures) if failures else 'none'

format_evidence['failed_reliable_gates'] = format_evidence.apply(failed_reliable_gates, axis=1)
format_evidence['format_use_explanation'] = np.where(
    format_evidence['used_in_official_score'],
    'Reliable format used in official score',
    format_evidence['evidence_status'] + ': excluded from official score; ' + format_evidence['failed_reliable_gates'],
)
for column in ['scoring_quality_z', 'survival_quality_z', 'scoring_pressure_gradient_z', 'survival_pressure_gradient_z',
               'wicket_quality_z', 'run_suppression_quality_z', 'wicket_pressure_gradient_z', 'run_suppression_pressure_gradient_z']:
    if column not in format_evidence:
        format_evidence[column] = np.nan
for column, (_, weight) in BAT_COMPONENTS.items():
    format_evidence[f'{column}_weighted_contribution'] = np.where(format_evidence['role'].eq('Batter'), format_evidence[column] * weight, np.nan)
for column, (_, weight) in BOWL_COMPONENTS.items():
    format_evidence[f'{column}_weighted_contribution'] = np.where(format_evidence['role'].eq('Bowler'), format_evidence[column] * weight, np.nan)
require(len(format_evidence) == 40, 'Expected exactly two format rows per official player')
require(format_evidence.loc[format_evidence['used_in_official_score'], 'evidence_status'].eq('Reliable').all(), 'Used format is not Reliable')
print(format_evidence.groupby(['role', 'format_group', 'evidence_status'], observed=True).size().to_string())

role    format_group  evidence_status    
Batter  IPL           Insufficient           2
                      No observed records    2
                      Reliable               6
        T20I          Provisional            3
                      Reliable               7
Bowler  IPL           No observed records    3
                      Reliable               7
        T20I          Insufficient           2
                      No observed records    3
                      Reliable               5


## Reconstruct high-pressure and ordinary outcome evidence

In [3]:
BENCHMARK_KEYS = ['format_group', 'innings', 'overid', 'structural_pressure_category']

def attach_expected(frame, metric):
    benchmark = outcome_benchmarks.loc[outcome_benchmarks['metric'].eq(metric), BENCHMARK_KEYS + ['expected_outcome']].copy()
    require(not benchmark.duplicated(BENCHMARK_KEYS).any(), f'Duplicate {metric} benchmark')
    result = frame.merge(benchmark, on=BENCHMARK_KEYS, how='left', validate='many_to_one')
    require(result['expected_outcome'].notna().all(), f'Missing {metric} expected outcome')
    return result

selected_batter_ids = set(official.loc[official['role'].eq('Batter'), 'player_id'])
selected_bowler_ids = set(official.loc[official['role'].eq('Bowler'), 'player_id'])
batting_mask = deliveries['striker_id'].isin(selected_batter_ids) & (deliveries['extras_type'].isna() | deliveries['extras_type'].isin(['byes', 'legbyes']))
batting = deliveries.loc[batting_mask].copy()
batting['dismissal_available'] = ~(batting['is_wicket'] & batting['dismissed_player_id'].isna())
batting['batter_dismissal'] = (batting['is_wicket'] & batting['dismissed_player_id'].notna() & batting['dismissed_player_id'].eq(batting['striker_id'])).astype(int)
bat_runs = attach_expected(batting, 'batter_runs')
bat_runs['residual'] = bat_runs['runs'] - bat_runs['expected_outcome']
bat_survival = attach_expected(batting.loc[batting['dismissal_available']].copy(), 'batter_dismissal')
bat_survival['residual'] = bat_survival['expected_outcome'] - bat_survival['batter_dismissal']

bowling = deliveries.loc[deliveries['bowler_id'].isin(selected_bowler_ids)].copy()
bowling['bowler_wicket'] = (bowling['is_wicket'] & bowling['dismissal_type_normalized'].isin(BOWLER_WICKET_TYPES)).astype(int)
bowl_wickets = attach_expected(bowling, 'bowler_wicket')
bowl_wickets['residual'] = bowl_wickets['bowler_wicket'] - bowl_wickets['expected_outcome']
bowl_runs = attach_expected(bowling.loc[bowling['extras_type'].isna()].copy(), 'bowler_batter_runs')
bowl_runs['residual'] = bowl_runs['expected_outcome'] - bowl_runs['runs']

def add_band(frame):
    result = frame.copy()
    result['pressure_band'] = np.where(result['structural_pressure_category'].eq('High'), 'High', 'Ordinary')
    return result

br = add_band(bat_runs).groupby(['striker_id', 'striker_name', 'format_group', 'pressure_band'], observed=True).agg(
    primary_records=('runs', 'size'), primary_raw_outcome=('runs', 'sum'), primary_residual_per100=('residual', lambda x: 100 * x.mean())
).reset_index().rename(columns={'striker_id': 'player_id', 'striker_name': 'player_name'})
bs = add_band(bat_survival).groupby(['striker_id', 'format_group', 'pressure_band'], observed=True).agg(
    secondary_records=('batter_dismissal', 'size'), secondary_raw_outcome=('batter_dismissal', 'sum'), secondary_residual_per100=('residual', lambda x: 100 * x.mean())
).reset_index().rename(columns={'striker_id': 'player_id'})
bat_pressure = br.merge(bs, on=['player_id', 'format_group', 'pressure_band'], how='left', validate='one_to_one')
bat_pressure['role'] = 'Batter'
bat_pressure['primary_metric'] = 'scoring'
bat_pressure['secondary_metric'] = 'survival'
bat_pressure['primary_raw_rate_per100'] = 100 * bat_pressure['primary_raw_outcome'] / bat_pressure['primary_records']
bat_pressure['secondary_raw_rate_per100'] = 100 * bat_pressure['secondary_raw_outcome'] / bat_pressure['secondary_records']

bw = add_band(bowl_wickets).groupby(['bowler_id', 'bowler_name', 'format_group', 'pressure_band'], observed=True).agg(
    primary_records=('bowler_wicket', 'size'), primary_raw_outcome=('bowler_wicket', 'sum'), primary_residual_per100=('residual', lambda x: 100 * x.mean())
).reset_index().rename(columns={'bowler_id': 'player_id', 'bowler_name': 'player_name'})
brr = add_band(bowl_runs).groupby(['bowler_id', 'format_group', 'pressure_band'], observed=True).agg(
    secondary_records=('runs', 'size'), secondary_raw_outcome=('runs', 'sum'), secondary_residual_per100=('residual', lambda x: 100 * x.mean())
).reset_index().rename(columns={'bowler_id': 'player_id'})
bowl_pressure = bw.merge(brr, on=['player_id', 'format_group', 'pressure_band'], how='left', validate='one_to_one')
bowl_pressure['role'] = 'Bowler'
bowl_pressure['primary_metric'] = 'wicket-taking'
bowl_pressure['secondary_metric'] = 'run suppression'
bowl_pressure['primary_raw_rate_per100'] = 100 * bowl_pressure['primary_raw_outcome'] / bowl_pressure['primary_records']
bowl_pressure['secondary_raw_rate_per100'] = 100 * bowl_pressure['secondary_raw_outcome'] / bowl_pressure['secondary_records']
pressure_evidence = pd.concat([bat_pressure, bowl_pressure], ignore_index=True, sort=False)
pressure_evidence['phase6_model_version'] = MODEL_VERSION
pressure_columns = ['role', 'player_id', 'player_name', 'format_group', 'pressure_band', 'primary_metric', 'primary_records', 'primary_raw_outcome', 'primary_raw_rate_per100', 'primary_residual_per100', 'secondary_metric', 'secondary_records', 'secondary_raw_outcome', 'secondary_raw_rate_per100', 'secondary_residual_per100', 'phase6_model_version']
pressure_evidence = pressure_evidence[pressure_columns].sort_values(['role', 'player_id', 'format_group', 'pressure_band']).reset_index(drop=True)
print(f'Reconstructed {len(pressure_evidence)} selected-player pressure-band rows.')

Reconstructed 64 selected-player pressure-band rows.


## Uncertainty joins and controlled player interpretations

In [4]:
bootstrap_parts = []
bootstrap_fields = ['ranked_frequency', 'rating_median', 'rating_p025', 'rating_p975', 'rank_median', 'rank_p025', 'rank_p975', 'top10_frequency', 'top10_monte_carlo_se']
for mode in ['fixed', 'nested']:
    part = bootstrap_summary.loc[bootstrap_summary['bootstrap_mode'].eq(mode), ['role', 'player_id'] + bootstrap_fields].copy()
    part = part.rename(columns={column: f'{column}_{mode}' for column in bootstrap_fields})
    bootstrap_parts.append(part)
uncertainty = bootstrap_parts[0].merge(bootstrap_parts[1], on=['role', 'player_id'], how='outer', validate='one_to_one')
scenario_stats = scenario_rankings.groupby(['role', 'player_id'], observed=True).agg(
    scenario_ranked_count=('scenario', 'nunique'), scenario_top10_count=('top10', 'sum')
).reset_index()
scenario_stats['scenario_top10_frequency'] = scenario_stats['scenario_top10_count'] / 13
class_lookup = boundary_source.loc[boundary_source['robustness_class'].eq('BOUNDARY_TOP10'), ['role', 'player_id', 'robustness_class']].drop_duplicates()
selected_evidence = (official.merge(uncertainty, on=['role', 'player_id'], how='left', validate='one_to_one')
                     .merge(scenario_stats, on=['role', 'player_id'], how='left', validate='one_to_one')
                     .merge(self_influence[['role', 'player_id', 'lopo_rank', 'lopo_rating', 'rating_change', 'rank_change', 'top10_status_changed']], on=['role', 'player_id'], how='left', validate='one_to_one')
                     .merge(class_lookup, on=['role', 'player_id'], how='left', validate='one_to_one'))

status_wide = format_evidence.pivot(index=['role', 'player_id'], columns='format_group', values=['evidence_status', 'failed_reliable_gates']).reset_index()
status_wide.columns = ['_'.join([str(x).lower() for x in col if str(x)]) if isinstance(col, tuple) else col for col in status_wide.columns]
status_wide = status_wide.rename(columns={'role_': 'role', 'player_id_': 'player_id'})
selected_evidence = selected_evidence.merge(status_wide, on=['role', 'player_id'], how='left', validate='one_to_one')

strength_rows = []
for (role, player_id), group in format_evidence.loc[format_evidence['used_in_official_score']].groupby(['role', 'player_id'], observed=True):
    components = BAT_COMPONENTS if role == 'Batter' else BOWL_COMPONENTS
    values = []
    for row in group.itertuples(index=False):
        for column, (label, weight) in components.items():
            value = getattr(row, column)
            if pd.notna(value):
                values.append((float(value) * weight, f'{row.format_group} {label}'))
    values.sort(key=lambda item: (item[0], item[1]))
    strength_rows.append({'role': role, 'player_id': player_id, 'weakest_metric': values[0][1], 'weakest_weighted_z': values[0][0], 'strongest_metric': values[-1][1], 'strongest_weighted_z': values[-1][0]})
selected_evidence = selected_evidence.merge(pd.DataFrame(strength_rows), on=['role', 'player_id'], validate='one_to_one')

def number(value, digits=2, signed=False):
    if pd.isna(value):
        return 'not available'
    return f'{value:+.{digits}f}' if signed else f'{value:.{digits}f}'

def selection_reason(row):
    return (f"Ranks #{int(row['rank'])} among {row['role'].lower()}s with an official combined rating of {row['combined_rating']:.2f}, "
            f"using reliable {row['qualifying_formats']} evidence. The strongest weighted component is {row['strongest_metric']} "
            f"({row['strongest_weighted_z']:+.2f} weighted z); the weakest is {row['weakest_metric']} ({row['weakest_weighted_z']:+.2f}).")

def format_interpretation(row):
    if int(row['qualifying_format_count']) == 2:
        difference = row['t20i_rating'] - row['ipl_rating']
        leader = 'T20I' if difference > 0 else 'IPL'
        return (f"IPL rating {row['ipl_rating']:.2f} and T20I rating {row['t20i_rating']:.2f} are combined 50/50. "
                f"{leader} is higher by {abs(difference):.2f} rating points.")
    used = row['qualifying_formats']
    other = 'T20I' if used == 'IPL' else 'IPL'
    used_rating = row['ipl_rating'] if used == 'IPL' else row['t20i_rating']
    status = row[f'evidence_status_{other.lower()}']
    reason = row[f'failed_reliable_gates_{other.lower()}']
    return (f"The official score is the {used} rating of {used_rating:.2f}. {other} evidence is {status} and is excluded because {reason}. "
            f"The excluded format is not assigned zero.")

def pressure_interpretation(row):
    fragments = []
    used_formats = str(row['qualifying_formats']).split('+')
    metric_rows = format_evidence.loc[(format_evidence['role'].eq(row['role'])) & (format_evidence['player_id'].eq(row['player_id']))]
    for fmt in used_formats:
        high = pressure_evidence.loc[(pressure_evidence['role'].eq(row['role'])) & (pressure_evidence['player_id'].eq(row['player_id'])) & (pressure_evidence['format_group'].eq(fmt)) & (pressure_evidence['pressure_band'].eq('High'))].iloc[0]
        ordinary = pressure_evidence.loc[(pressure_evidence['role'].eq(row['role'])) & (pressure_evidence['player_id'].eq(row['player_id'])) & (pressure_evidence['format_group'].eq(fmt)) & (pressure_evidence['pressure_band'].eq('Ordinary'))].iloc[0]
        metric = metric_rows.loc[metric_rows['format_group'].eq(fmt)].iloc[0]
        if row['role'] == 'Batter':
            fragments.append(
                f"{fmt}: high-pressure scoring residual {high['primary_residual_per100']:+.2f}/100 and survival residual {high['secondary_residual_per100']:+.2f}/100 versus context; "
                f"ordinary residuals {ordinary['primary_residual_per100']:+.2f} and {ordinary['secondary_residual_per100']:+.2f}; "
                f"continuous scoring/survival gradients {metric['scoring_pressure_gradient_raw_per100']:+.2f}/{metric['survival_pressure_gradient_raw_per100']:+.2f}."
            )
        else:
            fragments.append(
                f"{fmt}: high-pressure wicket residual {high['primary_residual_per100']:+.2f}/100 and run-suppression residual {high['secondary_residual_per100']:+.2f}/100 versus context; "
                f"ordinary residuals {ordinary['primary_residual_per100']:+.2f} and {ordinary['secondary_residual_per100']:+.2f}; "
                f"continuous wicket/run-suppression gradients {metric['wicket_pressure_gradient_raw_per100']:+.2f}/{metric['run_suppression_pressure_gradient_raw_per100']:+.2f}."
            )
    return ' '.join(fragments)

def uncertainty_interpretation(row):
    return (f"Top-10 inclusion is {row['top10_frequency_fixed']:.1%} in the fixed-benchmark bootstrap and {row['top10_frequency_nested']:.1%} in the nested bootstrap. "
            f"The player remains top 10 in {int(row['scenario_top10_count'])}/13 deterministic scenarios. "
            f"Leave-one-player-out benchmarking changes the rating by {row['rating_change']:+.3f} and rank by {int(row['rank_change']):+d}; top-10 membership does not change. "
            f"Classification: {row['robustness_class']}. Bootstrap inclusion measures ranking stability, not the probability of being objectively top 10.")

selected_evidence['selection_reason'] = selected_evidence.apply(selection_reason, axis=1)
selected_evidence['format_interpretation'] = selected_evidence.apply(format_interpretation, axis=1)
selected_evidence['pressure_interpretation'] = selected_evidence.apply(pressure_interpretation, axis=1)
selected_evidence['uncertainty_interpretation'] = selected_evidence.apply(uncertainty_interpretation, axis=1)
selected_evidence['phase6_model_version'] = MODEL_VERSION
uncertainty_columns = ['role', 'rank', 'player_id', 'player_name', 'combined_rating', 'robustness_class'] + [f'{field}_{mode}' for mode in ['fixed', 'nested'] for field in bootstrap_fields] + ['scenario_ranked_count', 'scenario_top10_count', 'scenario_top10_frequency', 'lopo_rating', 'lopo_rank', 'rating_change', 'rank_change', 'top10_status_changed', 'phase6_model_version']
uncertainty_output = selected_evidence[uncertainty_columns].copy()
print(selected_evidence[['role', 'rank', 'player_name', 'qualifying_formats', 'robustness_class']].to_string(index=False))

  role  rank        player_name qualifying_formats robustness_class
Batter     1    Nicholas Pooran                IPL   BOUNDARY_TOP10
Batter     2      Jimmy Neesham               T20I   BOUNDARY_TOP10
Batter     3      Hardik Pandya           IPL+T20I   BOUNDARY_TOP10
Batter     4        Jos Buttler                IPL   BOUNDARY_TOP10
Batter     5     Jaker Ali Anik               T20I   BOUNDARY_TOP10
Batter     6   Romario Shepherd               T20I   BOUNDARY_TOP10
Batter     7   Suryakumar Yadav           IPL+T20I   BOUNDARY_TOP10
Batter     8     Iftikhar Ahmed               T20I   BOUNDARY_TOP10
Batter     9       Shreyas Iyer                IPL   BOUNDARY_TOP10
Batter    10        Rinku Singh           IPL+T20I   BOUNDARY_TOP10
Bowler     1     Sufiyan Muqeem               T20I   BOUNDARY_TOP10
Bowler     2      Kuldeep Yadav           IPL+T20I   BOUNDARY_TOP10
Bowler     3       Sunil Narine                IPL   BOUNDARY_TOP10
Bowler     4     Mohammad Nawaz               T2

## Challenger and provisional appendix

In [5]:
candidate_base = boundary_source.loc[boundary_source['robustness_class'].isin(['CHALLENGER', 'PROVISIONAL_CHALLENGER'])].copy()
candidate_base = candidate_base.drop_duplicates(['role', 'player_id'])
official_keys = set(map(tuple, official[['role', 'player_id']].to_numpy()))
candidate_base = candidate_base.loc[~candidate_base[['role', 'player_id']].apply(tuple, axis=1).isin(official_keys)].copy()
candidate_base = candidate_base.drop(columns=['ranked_frequency_fixed', 'ranked_frequency_nested', 'top10_frequency_fixed', 'top10_frequency_nested', 'scenario_top10_count'], errors='ignore')
ranked_lookup = official_all[['role', 'player_id', 'rank', 'combined_rating', 'qualifying_formats']].rename(columns={'rank': '_primary_rank', 'combined_rating': '_primary_rating', 'qualifying_formats': '_primary_formats'})
candidate_base = candidate_base.merge(ranked_lookup, on=['role', 'player_id'], how='left', validate='one_to_one')
candidate_base['rank'] = candidate_base['rank'].fillna(candidate_base['_primary_rank'])
candidate_base['combined_rating'] = candidate_base['combined_rating'].fillna(candidate_base['_primary_rating'])
candidate_base['qualifying_formats'] = candidate_base['qualifying_formats'].fillna(candidate_base['_primary_formats'])
candidate_base['candidate_type'] = np.select(
    [candidate_base['robustness_class'].eq('CHALLENGER'), candidate_base['_primary_rank'].notna()],
    ['Reliable challenger', 'Reliable-ranked with provisional additional format'],
    default='Provisional-only challenger',
)
candidate_base = candidate_base.drop(columns=['_primary_rank', '_primary_rating', '_primary_formats'])
all_bs_parts = []
for mode in ['fixed', 'nested']:
    part = bootstrap_summary.loc[bootstrap_summary['bootstrap_mode'].eq(mode), ['role', 'player_id', 'top10_frequency', 'ranked_frequency']].copy()
    part = part.rename(columns={'top10_frequency': f'top10_frequency_{mode}', 'ranked_frequency': f'ranked_frequency_{mode}'})
    all_bs_parts.append(part)
candidate_uncertainty = all_bs_parts[0].merge(all_bs_parts[1], on=['role', 'player_id'], how='outer', validate='one_to_one')
chosen_format_rows = []
for candidate in candidate_base.itertuples(index=False):
    pool = format_metrics.loc[(format_metrics['role'].eq(candidate.role)) & (format_metrics['player_id'].eq(candidate.player_id))].copy()
    desired_status = 'Reliable' if candidate.robustness_class == 'CHALLENGER' else 'Provisional'
    preferred = pool.loc[pool['eligibility'].eq(desired_status)].copy()
    if preferred.empty:
        preferred = pool.copy()
    sort_column = 'format_rating' if desired_status == 'Reliable' and preferred['format_rating'].notna().any() else 'high_records'
    preferred = preferred.sort_values([sort_column, 'format_group'], ascending=[False, True], na_position='last')
    chosen_format_rows.append(preferred.iloc[0][['role', 'player_id', 'format_group', 'eligibility', 'high_records', 'ordinary_records', 'high_matches', 'high_clean_records', 'ordinary_clean_records']].to_dict())
best_format = pd.DataFrame(chosen_format_rows)
candidate_appendix = (candidate_base.merge(candidate_uncertainty, on=['role', 'player_id'], how='left', validate='one_to_one')
                      .merge(scenario_stats, on=['role', 'player_id'], how='left', validate='one_to_one')
                      .merge(best_format, on=['role', 'player_id'], how='left', validate='one_to_one'))
candidate_appendix = candidate_appendix.rename(columns={'format_group': 'best_evidence_format', 'eligibility': 'best_evidence_status'})
candidate_appendix['max_bootstrap_top10_frequency'] = candidate_appendix[['top10_frequency_fixed', 'top10_frequency_nested']].max(axis=1, skipna=True).fillna(0)
candidate_appendix['scenario_top10_count'] = candidate_appendix['scenario_top10_count'].fillna(0).astype(int)
candidate_appendix['scenario_ranked_count'] = candidate_appendix['scenario_ranked_count'].fillna(0).astype(int)
candidate_appendix['failed_reliable_gates'] = candidate_appendix.apply(
    lambda row: failed_reliable_gates(pd.Series({**row.to_dict(), 'evidence_status': row.get('best_evidence_status', 'No observed records')})), axis=1
)
candidate_appendix['_type_order'] = candidate_appendix['candidate_type'].map({'Reliable challenger': 0, 'Reliable-ranked with provisional additional format': 1, 'Provisional-only challenger': 2})
candidate_appendix = candidate_appendix.sort_values(
    ['role', '_type_order', 'max_bootstrap_top10_frequency', 'scenario_top10_count', 'high_records', 'player_id'],
    ascending=[True, True, False, False, False, True]
).drop(columns='_type_order').reset_index(drop=True)
candidate_appendix['phase6_model_version'] = MODEL_VERSION
print(candidate_appendix.groupby(['role', 'candidate_type'], observed=True).size().to_string())

role    candidate_type                                    
Batter  Provisional-only challenger                           43
        Reliable challenger                                   15
        Reliable-ranked with provisional additional format     9
Bowler  Provisional-only challenger                           55
        Reliable challenger                                   12
        Reliable-ranked with provisional additional format    12


## Validation, export and report

In [6]:
used_formats = format_evidence.loc[format_evidence['used_in_official_score']].copy()
recombined = used_formats.groupby(['role', 'player_id'], observed=True)['format_rating'].mean().rename('recombined_rating').reset_index()
score_check = official.merge(recombined, on=['role', 'player_id'], validate='one_to_one')
high_pressure = pressure_evidence.loc[pressure_evidence['pressure_band'].eq('High')].copy()
bat_recon = used_formats.loc[used_formats['role'].eq('Batter'), ['role', 'player_id', 'format_group', 'high_scoring_residual_raw_per100', 'high_survival_residual_raw_per100']].merge(
    high_pressure.loc[high_pressure['role'].eq('Batter'), ['role', 'player_id', 'format_group', 'primary_residual_per100', 'secondary_residual_per100']],
    on=['role', 'player_id', 'format_group'], validate='one_to_one'
)
bowl_recon = used_formats.loc[used_formats['role'].eq('Bowler'), ['role', 'player_id', 'format_group', 'high_wicket_residual_raw_per100', 'high_run_suppression_raw_per100']].merge(
    high_pressure.loc[high_pressure['role'].eq('Bowler'), ['role', 'player_id', 'format_group', 'primary_residual_per100', 'secondary_residual_per100']],
    on=['role', 'player_id', 'format_group'], validate='one_to_one'
)
residual_error = max(
    (bat_recon['high_scoring_residual_raw_per100'] - bat_recon['primary_residual_per100']).abs().max(),
    (bat_recon['high_survival_residual_raw_per100'] - bat_recon['secondary_residual_per100']).abs().max(),
    (bowl_recon['high_wicket_residual_raw_per100'] - bowl_recon['primary_residual_per100']).abs().max(),
    (bowl_recon['high_run_suppression_raw_per100'] - bowl_recon['secondary_residual_per100']).abs().max(),
)
checks = [
    ('upstream_gates_pass', phase3_gates['status'].eq('PASS').all() and phase4_gates['status'].eq('PASS').all() and phase5_gates['status'].eq('PASS').all(), 0, 'all upstream gates PASS'),
    ('official_selection_count', len(official) == 20 and official.groupby('role').size().eq(10).all(), official.groupby('role').size().to_dict(), '10 per role'),
    ('official_ids_unique', not official.duplicated(['role', 'player_id']).any(), int(official.duplicated(['role', 'player_id']).sum()), '0 duplicates'),
    ('selected_evidence_complete', len(selected_evidence) == 20, len(selected_evidence), 20),
    ('two_format_rows_per_player', len(format_evidence) == 40 and format_evidence.groupby(['role', 'player_id']).size().eq(2).all(), len(format_evidence), 40),
    ('used_formats_reliable', used_formats['evidence_status'].eq('Reliable').all(), int(used_formats['evidence_status'].ne('Reliable').sum()), '0 non-Reliable used formats'),
    ('excluded_formats_explained', format_evidence.loc[~format_evidence['used_in_official_score'], 'failed_reliable_gates'].ne('none').all(), int(format_evidence.loc[~format_evidence['used_in_official_score'], 'failed_reliable_gates'].eq('none').sum()), '0 unexplained excluded formats'),
    ('combined_ratings_reconciled', np.allclose(score_check['combined_rating'], score_check['recombined_rating'], rtol=0, atol=1e-7), float((score_check['combined_rating'] - score_check['recombined_rating']).abs().max()), '<=1e-7 export tolerance'),
    ('pressure_bands_complete', set(pressure_evidence['pressure_band']) == {'High', 'Ordinary'}, sorted(pressure_evidence['pressure_band'].unique()), 'High and Ordinary'),
    ('used_format_pressure_coverage', used_formats.merge(high_pressure[['role', 'player_id', 'format_group']], on=['role', 'player_id', 'format_group'], how='left', indicator=True)['_merge'].eq('both').all(), 0, 'all used formats have pressure evidence'),
    ('phase4_residuals_reproduced', residual_error <= 1e-6, float(residual_error), '<=1e-6'),
    ('bootstrap_evidence_complete', selected_evidence[['top10_frequency_fixed', 'top10_frequency_nested']].notna().all().all(), int(selected_evidence[['top10_frequency_fixed', 'top10_frequency_nested']].isna().sum().sum()), '0 missing'),
    ('scenario_count_locked', scenario_rankings['scenario'].nunique() == 13, scenario_rankings['scenario'].nunique(), 13),
    ('self_influence_complete', selected_evidence[['lopo_rank', 'lopo_rating']].notna().all().all(), int(selected_evidence[['lopo_rank', 'lopo_rating']].isna().sum().sum()), '0 missing'),
    ('all_official_boundary_disclosed', selected_evidence['robustness_class'].eq('BOUNDARY_TOP10').all(), selected_evidence['robustness_class'].value_counts().to_dict(), '20 BOUNDARY_TOP10'),
    ('narratives_complete', selected_evidence[['selection_reason', 'format_interpretation', 'pressure_interpretation', 'uncertainty_interpretation']].apply(lambda c: c.str.len().gt(0).all()).all(), 0, 'all four narratives complete'),
    ('appendix_excludes_official_top10', not set(map(tuple, candidate_appendix[['role', 'player_id']].to_numpy())) & set(map(tuple, official[['role', 'player_id']].to_numpy())), 0, 'no official top-10 overlap'),
    ('provisional_only_unranked', candidate_appendix.loc[candidate_appendix['candidate_type'].eq('Provisional-only challenger'), 'rank'].isna().all(), int(candidate_appendix.loc[candidate_appendix['candidate_type'].eq('Provisional-only challenger'), 'rank'].notna().sum()), '0 primary ranks'),
    ('additional_format_flags_ranked', candidate_appendix.loc[candidate_appendix['candidate_type'].eq('Reliable-ranked with provisional additional format'), 'rank'].notna().all(), int(candidate_appendix.loc[candidate_appendix['candidate_type'].eq('Reliable-ranked with provisional additional format'), 'rank'].isna().sum()), '0 missing primary ranks'),
    ('no_recent_weight', deliveries['phase3_model_version'].eq('structural_pressure_v1').all() and official_all['phase5_model_version'].eq('ranking_uncertainty_v1').all(), 'FULL_2022_2026_NO_RECENCY_WEIGHT', 'full period only'),
]
validation = pd.DataFrame(checks, columns=['check_name', 'passed', 'observed', 'acceptance_rule'])
validation['status'] = np.where(validation['passed'], 'PASS', 'FAIL')
validation['phase6_model_version'] = MODEL_VERSION
require(validation['passed'].all(), 'Phase 6 validation failed:\n' + validation.loc[~validation['passed']].to_string(index=False))
print(validation[['check_name', 'status']].to_string(index=False))

                      check_name status
             upstream_gates_pass   PASS
        official_selection_count   PASS
             official_ids_unique   PASS
      selected_evidence_complete   PASS
      two_format_rows_per_player   PASS
           used_formats_reliable   PASS
      excluded_formats_explained   PASS
     combined_ratings_reconciled   PASS
         pressure_bands_complete   PASS
   used_format_pressure_coverage   PASS
     phase4_residuals_reproduced   PASS
     bootstrap_evidence_complete   PASS
           scenario_count_locked   PASS
         self_influence_complete   PASS
 all_official_boundary_disclosed   PASS
             narratives_complete   PASS
appendix_excludes_official_top10   PASS
       provisional_only_unranked   PASS
  additional_format_flags_ranked   PASS
                no_recent_weight   PASS


In [7]:
format_evidence['phase6_model_version'] = MODEL_VERSION
selected_output = selected_evidence.sort_values(['role', 'rank']).reset_index(drop=True)
format_output = format_evidence.sort_values(['role', 'rank', 'format_group']).reset_index(drop=True)
pressure_output = pressure_evidence.copy()
uncertainty_output = uncertainty_output.sort_values(['role', 'rank']).reset_index(drop=True)
candidate_output = candidate_appendix.copy()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
artifacts = {
    'phase6_selected_player_evidence.csv': selected_output,
    'phase6_selected_format_evidence.csv': format_output,
    'phase6_pressure_vs_ordinary_evidence.csv': pressure_output,
    'phase6_uncertainty_and_robustness.csv': uncertainty_output,
    'phase6_challenger_and_provisional_appendix.csv': candidate_output,
    'phase6_validation_gates.csv': validation,
}
for name, frame in artifacts.items():
    frame.to_csv(OUTPUT_DIR / name, index=False, float_format='%.10g')

top_table = selected_output[['role', 'rank', 'player_name', 'qualifying_formats', 'combined_rating', 'top10_frequency_fixed', 'top10_frequency_nested', 'scenario_top10_count']].copy()
top_table = top_table.rename(columns={'player_name': 'player', 'qualifying_formats': 'formats', 'combined_rating': 'rating', 'top10_frequency_fixed': 'fixed_top10', 'top10_frequency_nested': 'nested_top10', 'scenario_top10_count': 'top10_scenarios'})
single_rows = []
for row in selected_output.loc[selected_output['qualifying_format_count'].eq(1)].itertuples(index=False):
    other = 'T20I' if row.qualifying_formats == 'IPL' else 'IPL'
    evidence_row = format_output.loc[(format_output['role'].eq(row.role)) & (format_output['player_id'].eq(row.player_id)) & (format_output['format_group'].eq(other))].iloc[0]
    single_rows.append({'role': row.role, 'player': row.player_name, 'used_format': row.qualifying_formats, 'excluded_format': other, 'status': evidence_row['evidence_status'], 'reason': evidence_row['failed_reliable_gates']})
single_table = pd.DataFrame(single_rows)

player_sections = []
for role in ['Batter', 'Bowler']:
    player_sections.append(f'## {role} evidence')
    for row in selected_output.loc[selected_output['role'].eq(role)].sort_values('rank').itertuples(index=False):
        player_sections.extend([
            f'### {int(row.rank)}. {row.player_name}',
            '',
            f'- **Selection:** {row.selection_reason}',
            f'- **Formats:** {row.format_interpretation}',
            f'- **Pressure evidence:** {row.pressure_interpretation}',
            f'- **Uncertainty:** {row.uncertainty_interpretation}',
            '',
        ])

challenger_tables = []
for role in ['Batter', 'Bowler']:
    for candidate_type in ['Reliable challenger', 'Reliable-ranked with provisional additional format', 'Provisional-only challenger']:
        view = candidate_output.loc[(candidate_output['role'].eq(role)) & (candidate_output['candidate_type'].eq(candidate_type))].head(5)
        if len(view):
            rendered = view[['player_name', 'candidate_type', 'rank', 'best_evidence_format', 'best_evidence_status', 'top10_frequency_fixed', 'top10_frequency_nested', 'scenario_top10_count', 'failed_reliable_gates']].rename(columns={'player_name': 'player'})
            challenger_tables.extend([f'### {role}: {candidate_type}s', '', markdown_table(rendered), ''])

report = f'''# Phase 6 — Evidence, Interpretation and Reporting

## Decision

The official selections remain the locked Phase 5 top 10 batters and top 10 bowlers, ranked by unrounded full-period combined rating. Phase 6 adds evidence and interpretation only; it does not change eligibility, scores, format combination, or rank.

All 20 official selections are classified as `BOUNDARY_TOP10` under the predefined requirement of at least 75% top-10 inclusion in both bootstrap modes. They remain official because bootstrap inclusion measures stability, not the probability that a player is objectively top 10.

## Official selections

{markdown_table(top_table)}

## Methodology used to interpret the rankings

- The official period is the complete eligible 2022–2026 dataset with no recent-form weighting.
- Pressure is assigned before the delivery outcome from wicket strain, recent wicket shock and innings-stage urgency. High pressure is the top within-format-and-innings 20% of the continuous index.
- Batters are evaluated on contextual scoring, survival and their pressure gradients. Bowlers are evaluated on contextual wicket-taking, run suppression and their pressure gradients.
- IPL and T20I are normalized independently. They are combined 50/50 only when both formats meet Reliable eligibility; an excluded format is never assigned zero.
- Outcome residuals compare the observed result with the expected result for format, innings, source over and pressure category. High-versus-ordinary tables are descriptive; the continuous gradients remain the direct pressure-response metrics.
- Uncertainty uses 5,000 fixed-benchmark and 1,000 nested-benchmark match-level replicates, 13 deterministic scenarios and leave-one-player-out benchmark sensitivity.

## Why some selections use only one format

{markdown_table(single_table)}

No observed record means no player-format performance row exists in the supplied analytical population. It does not support a claim about matches outside the supplied data.

{chr(10).join(player_sections)}
## Leading challengers and provisional candidates

Reliable challengers have a primary reliable-format rank but remain outside the official top 10. A reliable-ranked player with a provisional additional format retains that primary rank, while the provisional format stays excluded. Provisional-only challengers have no primary rating and must not be compared as if they passed the eligibility rules. The complete appendix is exported separately; the tables below show the first five under the locked ordering.

{chr(10).join(challenger_tables)}
## Limitations

- Structural pressure does not include target, required run rate, score parity, win probability or exact legal balls remaining.
- Low and Medium pressure are combined as Ordinary for the evidence tables; scoring still uses the continuous pressure percentile and contextual benchmarks.
- A delivery outcome does not create its pressure label. Runs, dots, wickets and survival are evaluated only after the pre-delivery state is fixed.
- Single-format ratings can rank highly, but the missing or unreliable format provides no balancing evidence.
- Bootstrap frequencies quantify sensitivity to match resampling and eligibility changes; they are not objective selection probabilities.
- The 75% core threshold identifies no official player as core, so all final explanations must retain the boundary qualification.

## Validation and Phase 7 handoff

All {len(validation)} Phase 6 validation gates passed. Phase 7 should preserve the official rank order, exact format-status explanations, boundary classifications and uncertainty language across the final submission and presentation.
'''
REPORT_PATH.write_text(report, encoding='utf-8')

manifest_rows = []
for name, frame in artifacts.items():
    path = OUTPUT_DIR / name
    manifest_rows.append({'artifact': name, 'rows': len(frame), 'columns': len(frame.columns), 'bytes': path.stat().st_size, 'sha256': sha256(path), 'model_version': MODEL_VERSION, 'source_sha256': sha256(PHASE3_DIR / 'phase3_structural_pressure_labelled.csv')})
manifest_rows.append({'artifact': REPORT_PATH.name, 'rows': len(report.splitlines()), 'columns': 1, 'bytes': REPORT_PATH.stat().st_size, 'sha256': sha256(REPORT_PATH), 'model_version': MODEL_VERSION, 'source_sha256': sha256(PHASE3_DIR / 'phase3_structural_pressure_labelled.csv')})
manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(OUTPUT_DIR / 'phase6_output_manifest.csv', index=False)
print(manifest.to_string(index=False))
print('PHASE6_COMPLETE')

                                               artifact  rows  columns  bytes                                                           sha256         model_version                                                    source_sha256
                    phase6_selected_player_evidence.csv    20       57  32413 a98c6518f80d3ce3bb3a78bb69077264b57cd3c6b0d70721995af36cd502f0fc evidence_reporting_v1 d6706b02c9e3ac916432319ae3309cf1daedd9a39feecbb93b8a0bb2680f7798
                    phase6_selected_format_evidence.csv    40       69  19819 41222de0a03f2b4f0db58ce1c0aaa4db447119a4cfcccc6a1fd36e3353e3333a evidence_reporting_v1 d6706b02c9e3ac916432319ae3309cf1daedd9a39feecbb93b8a0bb2680f7798
               phase6_pressure_vs_ordinary_evidence.csv    64       16   9520 3b914cd64399b87ab8d62abb762757592f9c4067a025c08ddc6868a0e3f41c98 evidence_reporting_v1 d6706b02c9e3ac916432319ae3309cf1daedd9a39feecbb93b8a0bb2680f7798
                  phase6_uncertainty_and_robustness.csv    20       33   6001 bd